# 07. Explanations

This notebook rebuilds, from the study's saved attributions:

- **Table S-XXXII** (`reports/table5_explain_summary.csv`, `table5_explain_agreement.csv`): share of each model's
  attribution on attacker-controllable features, and the agreement of the models' rankings (Kendall $\tau$);
- **Table S-XXXIII** (`reports/tables/drivers.csv`): each model's top drivers and the features whose attribution
  attacks change most;
- **Figures S8 to S12**: the SHAP and attribution figures, plus `fig_shap_xgboost_three` (the three-dataset version
  of Figure S8).

Inputs: `runs/<dataset>/explain/global_importance.csv`, `attack_shift.csv`, `agreement.csv`
(fraudadv.explain.attributions) and `shap_<family>.npz` (fraudadv.explain.shap_values), and
`data/processed/<dataset>/meta.json` (which features the attacker controls). These results also give the RQ4 numbers
of Section VI-H.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))   # the library (not needed after pip install -e .)
from fraudadv import notebook as nb
ROOT = nb.setup()          # works from the folder root, sets the study's environment

The output folders, created up front as `scripts/make_results.sh` does (`mkdir -p reports/tables`).

In [ ]:
(ROOT / "reports" / "tables").mkdir(parents=True, exist_ok=True)

## Table S-XXXII: what the models rely on

`make_report.explain_tables` writes `reports/table5_explain.md` (top drivers, attribution shift and agreement per
dataset) and then, through `explain_summary`, `reports/table5_explain_summary.csv` (per dataset and model: share of
the attribution mass on controllable and history features, top features, what successful attacks move) and
`reports/table5_explain_agreement.csv` (Kendall $\tau$ between models' rankings). Table S-XXXII is the
`share_controllable` column of the first, by dataset and model, and the $\tau$ columns of the second.

In [ ]:
from fraudadv.report import make_report as MR
from fraudadv.report import tables

MR.explain_tables()
E = nb.show_csv("reports/table5_explain_summary.csv", rows=8)
nb.show_csv("reports/table5_explain_agreement.csv")
E.pivot(index="model", columns="dataset", values="share_controllable").round(3)

## Table S-XXXIII: drivers and attack shift per dataset

`tables.drivers` reads `global_importance.csv` and `attack_shift.csv` of every dataset: per model, the three features
with the largest mean absolute attribution and, per threat setting, the three whose attribution changes most under
the high-confidence attack ($\varepsilon = 0.5$), into `reports/tables/drivers.csv`.

In [ ]:
tables.drivers().head(12)

## Figures S8 to S12: SHAP and attribution figures

`fraudadv.report.shap_figures.main` draws the beeswarm summaries of XGBoost, CatBoost and the MLP, the XGBoost
attribution shift under the constrained attack, and the attribution heatmap of every family into `reports/` (with the
numbers behind the last two in `reports/shap/`).

In [ ]:
from fraudadv.report import shap_figures

shap_figures.main()
nb.show_csv("reports/shap/shap_shift_xgboost_constrained.csv", rows=6)
nb.show_csv("reports/shap/attribution_share_top_features.csv", rows=6)

**Figure S8** (`fig_shap_xgboost.pdf`): XGBoost TreeSHAP summary per dataset.

In [ ]:
nb.show_figure("fig_shap_xgboost")

**Figure S9** (`fig_shap_catboost.pdf`): CatBoost TreeSHAP summary per dataset.

In [ ]:
nb.show_figure("fig_shap_catboost")

**Figure S10** (`fig_shap_shift.pdf`): XGBoost, constrained: mean TreeSHAP of the most-moved features before and after the attack.

In [ ]:
nb.show_figure("fig_shap_shift")

**Figure S11** (`fig_shap_mlp.pdf`): MLP Integrated Gradients summary per dataset.

In [ ]:
nb.show_figure("fig_shap_mlp")

**Figure S12** (`fig_attr_heatmap.pdf`): every family's share of global attribution per feature.

In [ ]:
nb.show_figure("fig_attr_heatmap")

## Run it yourself: TreeSHAP of the ULB XGBoost on 50 test rows

TreeSHAP values of the saved ULB XGBoost (seed 0) with the library's own `fraudadv.explain.attributions.attribute`
(exact TreeSHAP through XGBoost's `pred_contribs`), on 50 of the 1,000 test rows the stored attributions explain
(`explained_rows`: 25 of its frauds and 25 of its legits), in memory only. The values are compared with the stored
per-row values (`runs/ulb/explain/shap_xgboost.npz`), and the top features of the mean |SHAP| on these 50 rows with
the stored global importance (mean over all 1,000 rows, `global_importance.csv`). The 50-row ranking is an
illustration on a small sample, not the study's.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import kendalltau

from fraudadv.attacks.run_attacks import RUNS, context_ids, load_model
from fraudadv.data.load import load
from fraudadv.explain.attributions import attribute, explained_rows

a, meta = load("ulb")
idx = explained_rows(a["y_test"])                       # the 1,000 explained test rows: frauds first, then legits
pos = np.r_[0:25, len(idx) - 25:len(idx)]               # 25 frauds and 25 legits
rows = idx[pos]
m = load_model("ulb", "xgboost", 0)
phi = attribute(m, "xgboost", a["X_test"][rows], context_ids("ulb")["test"][rows])

stored = np.load(RUNS / "ulb" / "explain" / "shap_xgboost.npz")
assert np.array_equal(stored["idx"][pos], rows)
print(f"largest |live - stored| TreeSHAP value on the 50 rows: {np.abs(phi - stored['phi'][pos]).max():.2e}")

G = pd.read_csv(RUNS / "ulb" / "explain" / "global_importance.csv", index_col=0)["xgboost"]
live = pd.Series(np.abs(phi).mean(0), index=meta["features"])
print(f"Kendall tau, 50-row mean |SHAP| vs stored global importance: {kendalltau(live, G.reindex(live.index)).statistic:.3f}")
pd.DataFrame({"top features, 50 rows (live)": live.sort_values(ascending=False).index[:8],
              "mean |SHAP|, 50 rows": live.sort_values(ascending=False).values[:8].round(4),
              "top features, 1,000 rows (stored)": G.sort_values(ascending=False).index[:8],
              "global importance (stored)": G.sort_values(ascending=False).values[:8].round(4)})

## Rerun the runs behind these results from scratch

The calls `scripts/reproduce.sh` makes for the explanations (stage `extras`) on one dataset: per-row SHAP and
Integrated Gradients for the figures, then global importance, attack shift and agreement for every family. They take
hours on the large datasets and write into `runs/<dataset>/explain/` (and `reports/shap/`). Not run here
(`RERUN = False`).

In [ ]:
RERUN = False
DATASET = "ulb"

if RERUN:
    from fraudadv.explain import attributions, shap_values

    TAB = ["xgboost", "catboost", "mlp", "ft_transformer", "ivis"]
    CTX, FMS = ([], []) if DATASET in ("ulb", "baf") else (["hgnn", "hgt"], ["chronos2", "timesfm"])
    shap_values.run(DATASET)                                 # shap_<family>.npz for xgboost, catboost, mlp
    attributions.run(DATASET, TAB + CTX + FMS)               # global_importance.csv, attack_shift.csv, agreement.csv

## Check: every file this notebook wrote, against the reference build

In [ ]:
figures = ["fig_shap_xgboost", "fig_shap_xgboost_three", "fig_shap_catboost", "fig_shap_shift", "fig_shap_mlp",
           "fig_attr_heatmap"]
written = ["reports/table5_explain.md", "reports/table5_explain_summary.csv", "reports/table5_explain_agreement.csv",
           "reports/tables/drivers.csv",
           "reports/shap/shap_shift_xgboost_constrained.csv", "reports/shap/attribution_share_top_features.csv"]
written += [f"reports/{name}.{ext}" for name in figures for ext in ("pdf", "png")]
nb.check(written)